# Lab 2.4: Large Codebase Context Management

**What you'll build:** A scratchpad-based context management system for large codebase sessions — and learn why bigger context windows don't solve context degradation.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way — watch context degrade from specific references to vague generalities | 5 min |
| 2 | The Right Way — scratchpad files preserve context across turns and compaction | 5 min |
| 3 | Your Turn — build a scratchpad system for a multi-module refactoring task | 10 min |
| 4 | Stress Test — verify context survives simulated compaction and resumption | 5 min |

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

You're working on a large codebase refactoring. Over the course of many turns, you make architectural decisions, modify files, and track dependencies. The problem: after enough turns, the model starts losing track of specifics.

We'll simulate the context degradation curve and show how scratchpad files fix it.

In [ ]:
# Simulate a refactoring session with specific decisions
REFACTORING_DECISIONS = [
    {"turn": 1, "decision": "Rename PaymentGatewayService to PaymentProcessor in src/payments/gateway.py"},
    {"turn": 3, "decision": "Extract validation logic into new class PaymentValidator in src/payments/validator.py"},
    {"turn": 5, "decision": "Change OrderService.process() to accept PaymentProcessor instead of gateway string"},
    {"turn": 7, "decision": "Add event-driven pattern: PaymentProcessor emits PaymentCompleted event"},
    {"turn": 9, "decision": "Update ShippingService to listen for PaymentCompleted instead of polling"},
    {"turn": 11, "decision": "Deprecate legacy REST endpoint POST /api/v1/process-payment"},
    {"turn": 13, "decision": "Add migration script: migrate_payment_configs.py for database schema changes"},
    {"turn": 15, "decision": "Update API docs: new endpoint POST /api/v2/payments with event-driven response"}
]

# Build a long conversation history
def build_long_conversation(decisions, include_scratchpad=False):
    """Simulate a long codebase session."""
    messages = []
    
    for d in decisions:
        messages.append({"role": "user", "content": f"Let's do this: {d['decision']}"})
        messages.append({"role": "assistant", "content": f"Done. I've made that change. What's next?"})
        # Add filler turns to simulate long conversation
        for _ in range(2):
            messages.append({"role": "user", "content": "Show me the current state of that file."})
            messages.append({"role": "assistant", "content": "Here's the file content... [simulated output]"})
    
    # Final ask: summarize all decisions
    messages.append({"role": "user", "content": 
        "Please list ALL architectural decisions we've made in this session. "
        "Include specific class names, file paths, and endpoint URLs."})
    
    return messages

print(f"Simulating {len(REFACTORING_DECISIONS)} decisions over {len(REFACTORING_DECISIONS) * 3} turns")
print(f"\nDecisions to preserve:")
for d in REFACTORING_DECISIONS:
    print(f"  Turn {d['turn']}: {d['decision'][:70]}...")

---

## Phase 1: The Wrong Approach

Rely on conversation history alone. After 24+ turns, ask the model to recall specific decisions.

In [ ]:
messages = build_long_conversation(REFACTORING_DECISIONS)

system = "You are a senior software engineer helping with a codebase refactoring."

response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    system=system,
    messages=messages
)

naive_recall = response.content[0].text
print("=== NAIVE APPROACH: Decision Recall ===")
print(naive_recall)

In [ ]:
# Check which specific details were preserved
SPECIFIC_DETAILS = {
    "PaymentProcessor": "Renamed class name",
    "PaymentValidator": "Extracted validation class",
    "src/payments/validator.py": "New file path",
    "PaymentCompleted": "Event name",
    "ShippingService": "Updated listener service",
    "/api/v1/process-payment": "Deprecated endpoint",
    "migrate_payment_configs.py": "Migration script",
    "/api/v2/payments": "New endpoint"
}

def check_recall(text, details):
    results = {}
    for detail, description in details.items():
        results[detail] = detail in text
    return results

naive_checks = check_recall(naive_recall, SPECIFIC_DETAILS)
print("=== DETAIL PRESERVATION CHECK ===")
for detail, found in naive_checks.items():
    status = "FOUND" if found else "MISSING"
    desc = SPECIFIC_DETAILS[detail]
    print(f"  {status:>7}: {detail} ({desc})")

found = sum(naive_checks.values())
print(f"\nPreserved: {found}/{len(naive_checks)} specific details")
if found < len(naive_checks):
    print("Context degradation: specific names were lost or paraphrased.")

### Why details get lost

- **Context degradation curve:** After 20+ turns, the model references "the pattern we discussed" instead of `PaymentCompleted`.
- **Lost in the middle:** Decisions from turns 5-11 (middle of the conversation) get less attention.
- **Filler dilution:** The "show me the file" turns push important decisions further from the model's attention window.

---

## Phase 2: The Right Approach

Scratchpad files: externalize decisions to a persistent file that the model can re-read. This survives context compaction, session crashes, and subagent boundaries.

In [ ]:
# Build a scratchpad document from our decisions
def build_scratchpad(decisions):
    """Create a structured scratchpad from session decisions."""
    scratchpad = "# REFACTORING PROGRESS\n\n"
    scratchpad += "## Architectural Decisions\n\n"
    for d in decisions:
        scratchpad += f"- **Turn {d['turn']}:** {d['decision']}\n"
    scratchpad += "\n## Modified Files\n\n"
    scratchpad += "- src/payments/gateway.py (renamed class)\n"
    scratchpad += "- src/payments/validator.py (new file)\n"
    scratchpad += "- src/orders/service.py (updated dependency)\n"
    scratchpad += "- src/shipping/service.py (event listener)\n"
    scratchpad += "- scripts/migrate_payment_configs.py (new)\n"
    scratchpad += "- docs/api.md (new endpoint)\n"
    scratchpad += "\n## Key Names\n\n"
    scratchpad += "- Old class: PaymentGatewayService -> New: PaymentProcessor\n"
    scratchpad += "- New class: PaymentValidator\n"
    scratchpad += "- Event: PaymentCompleted\n"
    scratchpad += "- Deprecated: POST /api/v1/process-payment\n"
    scratchpad += "- New: POST /api/v2/payments\n"
    return scratchpad

scratchpad = build_scratchpad(REFACTORING_DECISIONS)
print("=== SCRATCHPAD CONTENT ===")
print(scratchpad)

In [ ]:
# Simulate a post-compaction session: fresh context + scratchpad
system_with_scratchpad = f"""You are a senior software engineer helping with a codebase refactoring.

The session was compacted. Here is the scratchpad with all decisions from the previous context:

```
{scratchpad}
```

Use this scratchpad as your source of truth for all decisions, file paths, and class names."""

# Fresh context — only the question, no conversation history
fresh_messages = [
    {"role": "user", "content": 
        "Please list ALL architectural decisions we've made in this session. "
        "Include specific class names, file paths, and endpoint URLs."}
]

response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    system=system_with_scratchpad,
    messages=fresh_messages
)

scratchpad_recall = response.content[0].text
print("=== SCRATCHPAD APPROACH: Decision Recall ===")
print(scratchpad_recall)

In [ ]:
# Compare both approaches
scratchpad_checks = check_recall(scratchpad_recall, SPECIFIC_DETAILS)

print("=" * 60)
print("COMPARISON: NAIVE vs SCRATCHPAD")
print("=" * 60)
print(f"{'Detail':<35} {'Naive':>10} {'Scratchpad':>10}")
print(f"{'-'*35} {'-'*10} {'-'*10}")
for detail in SPECIFIC_DETAILS:
    n = "FOUND" if naive_checks[detail] else "MISSING"
    s = "FOUND" if scratchpad_checks[detail] else "MISSING"
    print(f"{detail:<35} {n:>10} {s:>10}")

n_score = sum(naive_checks.values())
s_score = sum(scratchpad_checks.values())
total = len(SPECIFIC_DETAILS)
print(f"\n{'Total':<35} {n_score:>6}/{total} {s_score:>6}/{total}")

if s_score > n_score:
    print("\nScratchpad preserved more details — even after simulated compaction.")

---

## Phase 3: Your Turn

Build a scratchpad system for a multi-module migration task. You need to track decisions across 4 modules and ensure context survives compaction.

In [ ]:
MIGRATION_DECISIONS = {
    "auth_module": {
        "file": "src/auth/jwt_handler.py",
        "old_class": "JWTAuthenticator",
        "new_class": "TokenAuthService",
        "changes": ["Switch from HS256 to RS256", "Add refresh token rotation", "Deprecate /auth/legacy endpoint"]
    },
    "user_module": {
        "file": "src/users/profile_service.py",
        "old_class": "UserProfileManager",
        "new_class": "ProfileService",
        "changes": ["Add GDPR data export method", "Move avatar handling to MediaService", "Add soft-delete support"]
    },
    "notification_module": {
        "file": "src/notifications/dispatcher.py",
        "old_class": "NotificationSender",
        "new_class": "EventDispatcher",
        "changes": ["Replace polling with WebSocket push", "Add priority queue for critical alerts", "Max batch size: 100 events"]
    },
    "billing_module": {
        "file": "src/billing/subscription_handler.py",
        "old_class": "SubscriptionManager",
        "new_class": "BillingEngine",
        "changes": ["Add proration for mid-cycle upgrades", "New table: billing_events", "Stripe webhook: /hooks/stripe/v2"]
    }
}

REQUIRED_IN_RECALL = {
    "TokenAuthService": "auth class rename",
    "RS256": "algorithm change",
    "ProfileService": "user class rename",
    "GDPR": "compliance requirement",
    "EventDispatcher": "notification class rename",
    "WebSocket": "transport change",
    "BillingEngine": "billing class rename",
    "billing_events": "new table",
    "/hooks/stripe/v2": "webhook endpoint"
}

print(f"Migration task: {len(MIGRATION_DECISIONS)} modules")
for module, details in MIGRATION_DECISIONS.items():
    print(f"  {module}: {details['old_class']} -> {details['new_class']}")
print(f"\nDetails to preserve: {len(REQUIRED_IN_RECALL)}")

In [ ]:
# =============================================================
# TODO: Build a scratchpad from the migration decisions
# =============================================================
#
# Requirements:
# - Must include ALL class renames (old -> new)
# - Must include ALL file paths
# - Must include ALL specific changes (algorithm, table names, endpoints)
# - Should be structured for easy reference (not narrative prose)
#
# Think about:
# - What format is easiest for the model to reference? (Markdown? JSON?)
# - Should you group by module or by change type?
# - What section headers help the model find specific details?

YOUR_SCRATCHPAD = """
# TODO: Build your scratchpad content here.
# Include all decisions from MIGRATION_DECISIONS.
# Make it structured enough that a fresh context can use it.
"""

# Test: inject scratchpad into a fresh system prompt
your_system = f"""You are a senior software engineer continuing a migration task.
The session was compacted. Here is the scratchpad:

```
{YOUR_SCRATCHPAD}
```

Use this as your source of truth for all decisions."""

response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    system=your_system,
    messages=[{"role": "user", "content":
        "List ALL decisions across ALL modules. Include specific class names, "
        "file paths, table names, endpoints, and algorithm changes."}]
)

your_recall = response.content[0].text
print("=== YOUR SCRATCHPAD RECALL ===")
print(your_recall)

In [ ]:
# =============================================================
# Checker: validates your scratchpad
# =============================================================

your_checks = check_recall(your_recall, REQUIRED_IN_RECALL)

print("=== YOUR SCORECARD ===")
for detail, found in your_checks.items():
    status = "FOUND" if found else "MISSING"
    desc = REQUIRED_IN_RECALL[detail]
    print(f"  {status:>7}: {detail} ({desc})")

found_count = sum(your_checks.values())
total = len(REQUIRED_IN_RECALL)
print(f"\n  Preserved: {found_count}/{total}")

if found_count == total:
    print("\n  PASSED — all decisions preserved through simulated compaction!")
else:
    missing = [d for d, f in your_checks.items() if not f]
    print(f"\n  MISSING: {missing}")
    print("  Add these details to your scratchpad.")

---

## Phase 4: Stress Test

Verify your scratchpad works consistently across multiple fresh contexts.

In [ ]:
print("Running 5 fresh-context recalls from your scratchpad...\n")

results = []
for run in range(5):
    response = client.messages.create(
        model=MODEL,
        max_tokens=1000,
        system=your_system,
        messages=[{"role": "user", "content":
            "List ALL decisions. Include class names, file paths, table names, endpoints."}]
    )
    text = response.content[0].text
    checks = check_recall(text, REQUIRED_IN_RECALL)
    found = sum(checks.values())
    results.append(found)
    missing = [d for d, f in checks.items() if not f]
    print(f"  Run {run+1}: {found}/{total} preserved", end="")
    if missing:
        print(f" (missing: {missing})")
    else:
        print(" (perfect)")

print(f"\n=== CONSISTENCY ===")
print(f"Scores: {results}")
if all(r == total for r in results):
    print("Perfect consistency — your scratchpad reliably preserves all context.")
else:
    print("Some details were lost on some runs. Strengthen your scratchpad structure.")

### Key Takeaways

1. **Context degrades predictably.** Turns 1-10 accurate, 10-20 vague, 20+ generic. This is not a model flaw — it's an attention capacity constraint.
2. **Scratchpad files persist.** They survive context compaction (/compact), session crashes, and subagent delegation boundaries.
3. **Write to scratchpad BEFORE compacting.** /compact is lossy — externalize decisions to a file first.
4. **Structure matters.** A well-organized scratchpad (grouped by module, with specific names and paths) is more reliably referenced than narrative prose.
5. **Bigger context windows don't fix this.** The "lost in the middle" effect exists even in 200K context windows. Structural solutions are the fix.